# Step 4 — Combine, Compare & Include Classical ML
**Owner: whoever runs this last, once all three family notebooks (3a/3b/3c) are done**

*(Renamed from `03d` — folds in what used to be a separate "shared subset
comparison" notebook, since this one already has the shared 500-post LLM data
in hand.)*

Reads the per-model result files everyone already pushed to git — **does not
re-run any classification.** Run `git pull` first to make sure you have
everyone's latest `results/tables/multi_llm_*.csv` files, and that you have
`models/rf.joblib` locally (Drive-shared, not git-committed — see
docs/IMPLEMENTATION.md §5) if you want Random Forest included.

In [ ]:
import os
while not os.path.isdir("src"):
    os.chdir("..")
print("Working directory set to repo root:", os.getcwd())

In [ ]:
import sys
sys.path.append("src")

from multi_llm_eval import combine_saved_results, add_classical_ml_predictions, report_multi_model_metrics

## Combine everyone's LLM results
Prints which models are found and which are still missing (so you know who to ping).

In [ ]:
combined = combine_saved_results()

## Add classical ML into the same comparison

Re-evaluates the already-trained SVM (and Random Forest, if `rf.joblib` is
available) on the exact same posts every LLM was evaluated on — no
retraining, no new API calls. This is what used to be a separate
"shared subset comparison" step.

In [ ]:
augmented = add_classical_ml_predictions(combined)

## Compare everything — classical ML + every LLM, every family, one table

In [ ]:
metrics = report_multi_model_metrics(augmented)

## Chart for the presentation

In [ ]:
import matplotlib.pyplot as plt
import os
os.makedirs("results/figures", exist_ok=True)

colors = {
    "Classical ML": "#666666",
    "OpenAI": "#10a37f",
    "Google": "#4285f4",
    "Local (LM Studio)": "#cc8800",
}
bar_colors = [colors.get(f, "#333333") for f in metrics["family"]]

plt.figure(figsize=(10, 5))
plt.bar(metrics["model"], metrics["macro_f1"], color=bar_colors)
plt.ylabel("Macro-F1")
plt.title("Classical ML vs. zero-shot LLMs across 3 families")
plt.xticks(rotation=25, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig("results/figures/full_model_comparison.png", dpi=120)
plt.show()

## LLM rationale analysis
Mean rationale length and the most common confusion pairs, per LLM (classical
ML rows don't have rationales, so this only looks at LLM predictions).

In [ ]:
llm_only = augmented[augmented["family"] != "Classical ML"]
for model, group in llm_only.groupby("model"):
    wrong = group[group["true_label"] != group["predicted_label"]]
    if len(wrong):
        top_pair = (wrong["true_label"] + " -> " + wrong["predicted_label"]).value_counts().head(1)
        print(f"{model}: top confusion = {top_pair.index[0]} ({top_pair.iloc[0]} times)")

## Done
This is the headline result for the presentation — classical ML and every LLM, every family, one fixed evaluation set, one table.